In [ ]:
!pip install biopython requests pandas

In [ ]:
# @title ### **1. Setup & Installation**
!pip install biopython scikit-learn lime requests pandas numpy matplotlib seaborn -q
!pip install alphafold
!pip install colabfold
import pandas as pd
import numpy as np
from Bio.Blast import NCBIWWW
from Bio import SeqIO
import requests
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
import lime
import lime.lime_tabular
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from io import StringIO
import os

In [ ]:
import requests
import pandas as pd
from io import StringIO
from time import sleep

def fetch_uniprot_data(query="reviewed:true", limit=100, max_retries=3):
    """
    Fetches protein data from UniProt's current REST API

    Args:
        query (str): Search query (default: reviewed proteins)
        limit (int): Maximum number of results
        max_retries (int): Number of retry attempts

    Returns:
        pd.DataFrame with columns: uniprot_id, sequence, function
    """
    base_url = "https://rest.uniprot.org/uniprotkb/search"

    params = {
        'query': query,
        'fields': 'accession,sequence,protein_name',
        'size': limit,
        'format': 'tsv'
    }

    headers = {
        'Accept': 'text/plain',
        'User-Agent': 'Python-Script'
    }

    for attempt in range(max_retries):
        try:
            response = requests.get(base_url, params=params, headers=headers)
            response.raise_for_status()

            # Parse TSV data
            df = pd.read_csv(StringIO(response.text), sep='\t')
            df.columns = ['uniprot_id', 'sequence', 'function']

            # Validate data
            if df.empty:
                raise ValueError("Empty response - check your query")
            if pd.isna(df['sequence']).any():
                raise ValueError("Missing sequence data")

            return df

        except Exception as e:
            print(f"Attempt {attempt+1} failed: {str(e)}")
            if attempt == max_retries - 1:
                raise
            sleep(5)  # Wait before retry

    return pd.DataFrame(columns=['uniprot_id', 'sequence', 'function'])

# Usage example
try:
    print("Fetching protein data...")
    protein_df = fetch_uniprot_data(limit=100)
    print(f"Success! Downloaded {len(protein_df)} proteins")
    print(protein_df.head())

except Exception as e:
    print(f"Failed to fetch data: {str(e)}")

In [ ]:
# @title ### **1. INSTALL REQUIRED TOOLS**
!apt update && apt install -y diamond-aligner
!wget https://mmseqs.com/foldseek/foldseek-linux-avx2.tar.gz
!tar xvfz foldseek-linux-avx2.tar.gz
!mv foldseek/bin/foldseek /usr/local/bin/
!rm -rf foldseek*a

# Install AlphaFold dependencies (if not already present)
!pip install --upgrade pip
!pip install alphafold-colabfold

# @title ### **2. SETUP DATABASES**
import os
from pathlib import Path

# Create working directories
Path("databases").mkdir(exist_ok=True)
Path("tmp").mkdir(exist_ok=True)

# Download SwissProt (for Diamond BLAST)
if not os.path.exists("uniprot_sprot.fasta"):
    print("Downloading SwissProt...")
    !wget https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/uniprot_sprot.fasta.gz
    !gunzip uniprot_sprot.fasta.gz
    !diamond makedb --in uniprot_sprot.fasta -d databases/swissprot --threads 4

# Download Foldseek example database
if not os.path.exists("databases/example"):
    print("Downloading Foldseek example database...")
    !wget -O example_db.tar.gz https://mmseqs.com/foldseek/example_db.tar.gz
    !tar xvfz example_db.tar.gz -C databases/
    !rm example_db.tar.gz

# @title ### **3. FEATURE EXTRACTION PIPELINE**
import pandas as pd
import numpy as np
import subprocess
from concurrent.futures import ThreadPoolExecutor
from colabfold import alphafold

class ProteinFeatureExtractor:
    def __init__(self):
        self.validate_databases()

    def validate_databases(self):
        """Verify all required databases exist"""
        required_files = {
            'Diamond': 'databases/swissprot.dmnd',
            'Foldseek': 'databases/example'
        }
        for name, path in required_files.items():
            if not os.path.exists(path):
                raise FileNotFoundError(f"{name} database not found at {path}")

    def run_command(self, cmd):
        """Execute command with error handling"""
        try:
            result = subprocess.run(
                cmd, shell=True, check=True,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True
            )
            return True, result.stdout
        except subprocess.CalledProcessError as e:
            print(f"Command failed: {e.cmd}")
            print(f"Error: {e.stderr}")
            return False, None

    def extract_blast_features(self, sequence, tmp_prefix):
        """Extract sequence similarity features using Diamond BLAST"""
        query_file = f"{tmp_prefix}.fa"
        blast_out = f"{tmp_prefix}_blast.out"

        features = {'blast_evalue': 0, 'blast_identity': 0}

        with open(query_file, 'w') as f:
            f.write(f">query\n{sequence}\n")

        cmd = (
            f"diamond blastp -q {query_file} -d databases/swissprot "
            f"--outfmt 6 qseqid sseqid pident evalue --threads 2 "
            f"--max-target-seqs 1 > {blast_out}"
        )
        success, _ = self.run_command(cmd)

        if success and os.path.getsize(blast_out) > 0:
            try:
                blast_results = pd.read_csv(blast_out, sep='\t', header=None)
                if not blast_results.empty:
                    features['blast_identity'] = float(blast_results.iloc[0, 2])
                    evalue = float(blast_results.iloc[0, 3])
                    features['blast_evalue'] = -np.log10(evalue) if evalue > 0 else 0
            except Exception as e:
                print(f"BLAST results processing error: {e}")

        for f in [query_file, blast_out]:
            if os.path.exists(f):
                os.remove(f)
        return features

    def extract_foldseek_features(self, sequence, tmp_prefix):
        """Extract structural similarity features using Foldseek"""
        query_file = f"{tmp_prefix}.fa"
        fs_out = f"{tmp_prefix}_fs.out"

        features = {'fs_tmscore': 0}

        with open(query_file, 'w') as f:
            f.write(f">query\n{sequence}\n")

        cmd = (
            f"foldseek easy-search {query_file} databases/example "
            f"{fs_out} tmp --threads 2 --max-accept 1"
        )
        success, _ = self.run_command(cmd)

        if success and os.path.getsize(fs_out) > 0:
            try:
                fs_results = pd.read_csv(fs_out, sep='\t', header=None)
                if not fs_results.empty and len(fs_results.columns) > 10:
                    features['fs_tmscore'] = float(fs_results.iloc[0, 10])
            except Exception as e:
                print(f"Foldseek results processing error: {e}")

        for f in [query_file, fs_out]:
            if os.path.exists(f):
                os.remove(f)
        return features

    def run_alphafold(self, sequence, tmp_prefix):
        """Predict structure using AlphaFold (via ColabFold)"""
        features = {'af_plddt': 0, 'af_ptm': 0}

        try:
            # Run AlphaFold prediction
            result = alphafold.predict_structure(
                sequence,
                f"{tmp_prefix}_af",  # Output prefix
                model_type="auto"
            )

            # Extract features (pLDDT and predicted TM-score)
            features['af_plddt'] = result['plddt'].mean()
            features['af_ptm'] = result['ptm']

        except Exception as e:
            print(f"AlphaFold prediction error: {e}")

        return features

    def extract_features(self, sequence):
        """Extract all features (BLAST + Foldseek + AlphaFold)"""
        tmp_prefix = f"tmp/seq_{os.getpid()}_{hash(sequence)}"

        # Get features from all tools
        blast_features = self.extract_blast_features(sequence, tmp_prefix)
        fs_features = self.extract_foldseek_features(sequence, tmp_prefix)
        af_features = self.run_alphafold(sequence, tmp_prefix)

        # Combine all features
        combined_features = {
            **blast_features,
            **fs_features,
            **af_features
        }

        return combined_features

In [ ]:
# 1. Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 2. Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# Feature names for interpretation
feature_names = [
    'blast_evalue',
    'blast_identity',
    'af_plddt',
    'af_helix',
    'foldseek_tmscore',
    'foldseek_aligned'
]

In [ ]:
# Optimal parameters from your ablation study
model = KNeighborsClassifier(
    n_neighbors=7,
    metric='cosine',
    weights='distance'
)
model.fit(X_train, y_train)

# Evaluate
accuracy = model.score(X_test, y_test)
print(f"Model Accuracy: {accuracy:.1%}")

In [ ]:
# Initialize explainer
explainer = lime.lime_tabular.LimeTabularExplainer(
    X_train,
    feature_names=feature_names,
    class_names=np.unique(y_train),
    mode='classification'
)

# Explain a prediction
def explain_protein(sequence):
    features = extract_features(sequence)
    if not features:
        print("Feature extraction failed")
        return

    scaled_features = scaler.transform([list(features.values())])
    exp = explainer.explain_instance(
        scaled_features[0],
        model.predict_proba,
        num_features=len(feature_names)
    )
    return exp

# Example usage:
user_sequence = get_user_sequence()
if user_sequence:
    explanation = explain_protein(user_sequence)
    explanation.show_in_notebook()

In [ ]:
sequence = get_user_sequence()
if sequence:
    # 1. Get features
    features = extract_features(sequence)

    if features:
        # 2. Scale and predict
        scaled_features = scaler.transform([list(features.values())])
        prediction = model.predict(scaled_features)[0]
        confidence = model.predict_proba(scaled_features).max()

        print(f"\nPredicted Function: {prediction} (Confidence: {confidence:.1%})")

        # 3. Explain
        explanation = explain_protein(sequence)
        explanation.show_in_notebook()
    else:
        print("Feature extraction failed")
else:
    print("Invalid sequence")

In [ ]:
# Add these at the top of your imports
!pip install lime
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import lime
import lime.lime_tabular
import numpy as np
import pandas as pd

In [ ]:
def create_feature_vector(blast_results, foldseek_hits, alphafold_structure):
    """Convert existing outputs into a feature vector"""
    features = {}

    # 1. BLASTp Features (from your parse_blast_results)
    if blast_results:
        features.update({
            'blast_evalue': -np.log10(blast_results[0][1]),  # First hit's E-value
            'blast_identity': blast_results[0][2] if len(blast_results[0]) > 2 else 0  # % identity if available
        })

    # 2. FoldSeek Features (from parse_foldseek_m8)
    if foldseek_hits:
        features.update({
            'foldseek_tmscore': foldseek_hits[0]['tm_score'],
            'foldseek_aligned': foldseek_hits[0]['aligned_length'] / 100.0  # Normalized
        })

    # 3. AlphaFold Features (from fetch_structure)
    if alphafold_structure:
        # Mock values - replace with actual pLDDT/helix extraction
        features.update({
            'af_plddt': 85.0,  # Replace with real mean pLDDT
            'af_helix': 0.25    # Replace with DSSP helix%
        })

    return features

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
sequence = get_user_sequence()

if sequence is None:
    print("Protein validation failed. Exiting program.")
else:
    blast_results_xml = run_blastp(sequence)

    if blast_results_xml:

        protein_list, _ = parse_blast_results(blast_results_xml)

        if protein_list:
            print("\nList of similar proteins with E-values:")
            for prot_id, e_value in protein_list:
                print(f"UniProt ID: {prot_id}, E-value: {e_value}")
    else:
        print("No BLAST results received. Please try again later.")

DATA EXTRACTION

In [ ]:
from proteininput import get_user_sequence
from pblast import run_blastp, parse_blast_results
from pdbstructure import search_pdb_by_sequence, fetch_pdb_structure
from alphafold import get_alphafold_structure, fetch_structure
from foldseek import parse_foldseek_m8, display_results
import os
import requests

sequence = get_user_sequence()

if sequence is None:
    print("Protein validation failed. Exiting program.")
else:
    blast_results_xml = run_blastp(sequence)

    if blast_results_xml:

        protein_list, _ = parse_blast_results(blast_results_xml)

        if protein_list:
            print("\nList of similar proteins with E-values:")
            for prot_id, e_value in protein_list:
                print(f"UniProt ID: {prot_id}, E-value: {e_value}")
    else:
        print("No BLAST results received. Please try again later.")


file_path = input("Enter the path to your Foldseek .m8 file: ").strip()

evalue_input = input("Enter the E-value threshold (default 0.001): ").strip()
try:
        evalue_threshold = float(evalue_input) if evalue_input else 0.001
except ValueError:
        print("Invalid input. Using default threshold of 0.001.")
        evalue_threshold = 0.001

hits = parse_foldseek_m8(file_path, evalue_threshold)
display_results(hits)


uniprot_id = input("Enter the UniProt ID (e.g., Q9I579): ").strip()
structure_file = fetch_structure(uniprot_id)

if structure_file:
    print(f"Structure saved in file: {structure_file}")
else:
    print("No structure could be fetched.")
